# 02 - Baseline Models para Prediccion de Retrasos

clasificar `Late_delivery_risk` usando dos algoritmos: Regresion Logistica y Random Forest.

## 1. Objetivo y alcance

1. Preparar datos sin fuga de informacion.
2. Entrenar dos modelos baseline.
3. Comparar metricas: accuracy, precision, recall, f1, roc_auc.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = Path('../data/raw/DataCoSupplyChainDataset.csv')
df = pd.read_csv(DATA_PATH, encoding='latin-1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head(3)

Dataset: 180519 filas x 53 columnas


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [3]:
target_col = 'Late_delivery_risk'

leakage_cols = [
    'Delivery Status',
    'shipping date (DateOrders)',
    'Order Status',
    'Order Id',
    'Order Item Id',
    'Customer Email',
    'Customer Password',
    'Product Description',
]

drop_cols = [c for c in leakage_cols if c in df.columns]
model_df = df.drop(columns=drop_cols).copy()

if target_col not in model_df.columns:
    raise ValueError('No se encontro Late_delivery_risk en el dataset.')

X = model_df.drop(columns=[target_col])
y = model_df[target_col].astype(int)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print(f'Variables numericas: {len(num_cols)}')
print(f'Variables categoricas: {len(cat_cols)}')
print(f'Variables removidas por posible fuga: {drop_cols}')

Variables numericas: 25
Variables categoricas: 19
Variables removidas por posible fuga: ['Delivery Status', 'shipping date (DateOrders)', 'Order Status', 'Order Id', 'Order Item Id', 'Customer Email', 'Customer Password', 'Product Description']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
    ]
)

models = {
    'logistic_regression': LogisticRegression(max_iter=1200, class_weight='balanced'),
    'random_forest': RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample'
    )
}

results = []
trained = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('prep', preprocessor),
        ('model', clf)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    })

    trained[name] = {'pipeline': pipe, 'y_pred': y_pred, 'y_prob': y_prob}

metrics_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
metrics_df

,model,accuracy,precision,recall,f1,roc_auc
0,random_forest,0.987674,0.978157,0.999848,0.988884,0.998755
1,logistic_regression,0.979587,0.976213,0.986816,0.981486,0.992783


In [5]:
best_model_name = metrics_df.loc[0, 'model']
best = trained[best_model_name]

print(f'Mejor baseline segun F1: {best_model_name}')
print('\nReporte de clasificacion:')
print(classification_report(y_test, best['y_pred']))

cm = confusion_matrix(y_test, best['y_pred'])
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
cm_df

Mejor baseline segun F1: random_forest

Reporte de clasificacion:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     16308
           1       0.98      1.00      0.99     19796

    accuracy                           0.99     36104
   macro avg       0.99      0.99      0.99     36104
weighted avg       0.99      0.99      0.99     36104



,Pred 0,Pred 1
Real 0,15866,442
Real 1,3,19793


## 2. Conclusiones baseline

1. Se construyo un flujo con separacion entrenamiento-prueba y transformaciones por tipo de variable.
2. Se compararon dos algoritmos baseline sobre las mismas particiones y metricas.
3. El mejor modelo debe pasar a una fase de ajuste de hiperparametros y validacion antes de despliegue.